In [1]:
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.metrics import classification_report, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# File path to your claims dataset
file_path = r"C:\Users\Sujin Andro\OneDrive\Desktop\movies\rcm claim\claims_main.csv"

df = pd.read_csv(file_path)
print(f"Dataset loaded. Shape: {df.shape}")
df.head()

Dataset loaded. Shape: (120000, 25)


,claim_id,claim_submission_date,claim_year,claim_quarter,payer_type,provider_specialty,place_of_service_code,place_of_service_desc,cpt_code,modifier,...,prior_auth_obtained,prior_auth_number,documentation_completeness,claim_amount_usd,outcome,denial_reason_code,denial_category,dataset_version,synthetic_flag,generation_date
0,e46071e9-0fe0-42d8-ab46-dbb0c48341c4,2022-04-22,2022,Q2,Commercial_PPO,Emergency_Medicine,23,Emergency_Room,99283,NaN,...,NaN,NaN,0.40,656.22,denied,CO-4,coding_error,2.0,True,2026-04-28
1,a641a4ce-38d4-4467-a9d3-d4970735ecc0,2022-07-24,2022,Q3,Medicare_FFS,Internal_Medicine,11,Office,99215,25,...,NaN,NaN,0.37,332.89,denied,CO-97,duplicate,2.0,True,2026-04-28
2,2bd53273-5f36-46ad-b777-0035afd1e2e7,2022-08-24,2022,Q3,Commercial_EPO,Cardiology,2,Telehealth,93000,NaN,...,NaN,NaN,0.65,185.97,denied,CO-4,bundling,2.0,True,2026-04-28
3,72df5380-e03a-4ae9-a1e0-3d3aadf13daf,2021-07-26,2021,Q3,Commercial_PPO,Physical_Therapy,11,Office,97530,59,...,NaN,NaN,0.85,157.25,paid,NaN,NaN,2.0,True,2026-04-28
4,9eb5d58c-3f62-40a9-971f-37855af76d48,2021-12-01,2021,Q4,Medicaid_Managed,Obstetrics_Gynecology,23,Emergency_Room,59400,NaN,...,False,NaN,0.38,8991.46,denied,CO-16,auth_missing,2.0,True,2026-04-28


In [4]:
# Verify outcome column exists
if "outcome" not in df.columns:
    raise KeyError("Column 'outcome' not found in dataset. Please check your column names.")

# Map target: Denied = 1, Paid/Approved = 0
if df["outcome"].dtype == object:
    df["target"] = (
        df["outcome"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"denied": 1, "paid": 0, "approved": 0})
        .fillna(0)
    )
else:
    df["target"] = df["outcome"].astype(int)

print("Target variable mapped successfully!")
print(df["target"].value_counts(normalize=True))

Target variable mapped successfully!
target
0.0    0.719467
1.0    0.280533
Name: proportion, dtype: float64


In [5]:
# Select pre-adjudication features (preventing data leakage)
feature_cols = [
    "payer_type",
    "provider_place_of_service",
    "place_of_service",
    "cpt_code",
    "modifier",
    "primary_icd",
    "secondary_icd",
    "prior_auth_required",
    "prior_auth_submitted",
    "prior_auth_approved",
    "claim_amount",
]

# Keep only columns that exist in the dataset
feature_cols = [col for col in feature_cols if col in df.columns]
print(f"Selected features: {feature_cols}")

X = df[feature_cols].copy()
y = df["target"]

# Categorical Encoding & Handling Missing Values
encoders = {}
for col in X.columns:
    if X[col].dtype == object or X[col].dtype.name == "category":
        X[col] = X[col].fillna("Missing").astype(str)
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        encoders[col] = le
    else:
        X[col] = X[col].fillna(0)

print("Features preprocessed successfully!")

Selected features: ['payer_type', 'cpt_code', 'modifier', 'prior_auth_required']
Features preprocessed successfully!


In [7]:
# 80/20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Compute class imbalance ratio for XGBoost
pos_weight = (
    (len(y_train) - sum(y_train)) / sum(y_train) if sum(y_train) > 0 else 1.0
)
print(f"Dataset split complete! Scale Pos Weight calculated: {pos_weight:.2f}")

Dataset split complete! Scale Pos Weight calculated: 2.56


In [8]:
# Train XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
)

model.fit(X_train, y_train)

# Predict probabilities and apply recall-focused threshold (0.35)
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.35).astype(int)

print("\n--- Model Evaluation (Threshold = 0.35) ---")
print(classification_report(y_test, y_pred))
print(f"Overall Recall Score: {recall_score(y_test, y_pred):.2%}")


--- Model Evaluation (Threshold = 0.35) ---
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00     17267
         1.0       0.28      1.00      0.44      6733

    accuracy                           0.28     24000
   macro avg       0.14      0.50      0.22     24000
weighted avg       0.08      0.28      0.12     24000

Overall Recall Score: 100.00%


C:\Users\Sujin Andro\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sujin Andro\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Sujin Andro\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [9]:
# Save model, feature names, and encoders for app.py
joblib.dump(model, "claim_denial_model.joblib")
joblib.dump(feature_cols, "feature_names.joblib")
joblib.dump(encoders, "label_encoders.joblib")

# Extract dropdown options for the Streamlit UI
ui_options = {col: encoders[col].classes_.tolist() for col in encoders}
joblib.dump(ui_options, "ui_options.joblib")

print("All artifacts (.joblib) successfully saved to working directory!")


All artifacts (.joblib) successfully saved to working directory!


In [1]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd
import numpy as np

# Set page title
st.set_page_config(page_title="RCM Claim Denial Predictor", layout="wide")

st.title("🏥 Healthcare RCM - Claim Denial Risk Predictor")
st.write("Predict whether a claim will be denied prior to submission to reduce administrative cost and improve clean claim rates.")

# Load saved artifacts
@st.cache_resource
def load_artifacts():
    model = joblib.load("claim_denial_model.joblib")
    feature_names = joblib.load("feature_names.joblib")
    label_encoders = joblib.load("label_encoders.joblib")
    ui_options = joblib.load("ui_options.joblib")
    return model, feature_names, label_encoders, ui_options

try:
    model, feature_names, label_encoders, ui_options = load_artifacts()
    st.success("Model artifacts loaded successfully!")
except Exception as e:
    st.error(f"Error loading model files: {e}")
    st.stop()

# --- INPUT FORM ---
st.subheader("📋 Enter Claim Information")

# Example UI Inputs (Adjust column names to match your dataset)
col1, col2 = st.columns(2)

with col1:
    st.markdown("### Claim Details")
    # Add input fields based on your ui_options or feature names
    # e.g., prior_auth = st.selectbox("Prior Auth Status", ui_options.get('prior_auth_status', ['Yes', 'No']))

with col2:
    st.markdown("### Financials")
    # e.g., claim_amount = st.number_input("Claim Amount ($)", min_value=0.0, value=1500.0)

# Predict Button
if st.button("Evaluate Claim Risk"):
    # Preprocess inputs and run model.predict_proba()
    # Apply your recall-focused threshold of 0.35
    threshold = 0.35
    
    # Placeholder logic for example:
    st.warning("Prediction pipeline executing...")

Writing app.py


In [2]:
%%writefile app.py
import streamlit as st
import joblib

st.title("🏥 RCM Claim Denial Predictor")
st.write("App loaded successfully!")

Overwriting app.py
